# Comparing genomes with JBrowseR in Google Colab

[JBrowseR](https://github.com/GMOD/JBrowseR) renders the GPU-accelerated [JBrowse 2](https://jbrowse.org/jb2/) genome browser from R. Where `JBrowseR()` shows one linear view, `JBrowseRApp()` drives the whole app, so a `views` list can hold a linear view, a synteny view, or a dotplot.

This notebook stacks four *E. coli* strains tied by a single all-vs-all minimap2 alignment — the hosted data from the JBrowse [all-vs-all synteny tutorial](https://jbrowse.org/jb2/docs/tutorials/allvsall_synteny/). Run it on Colab's **R runtime** (Runtime -> Change runtime type -> R).


## Install

Installs JBrowseR from GitHub. The package ships a prebuilt JavaScript bundle, so there is no JavaScript toolchain to set up.

In [ ]:
if (!requireNamespace("JBrowseR", quietly = TRUE)) {
  install.packages("remotes")
  remotes::install_github("GMOD/JBrowseR")
}

## A helper to show a browser inline

Colab renders R htmlwidgets most reliably inside a sandboxed iframe, so this helper saves a widget to self-contained HTML and embeds it.

In [ ]:
library(JBrowseR)
library(htmlwidgets)
library(IRdisplay)

show_browser <- function(widget, height = 480) {
  f <- tempfile(fileext = ".html")
  saveWidget(widget, f, selfcontained = TRUE)
  html <- paste(readLines(f, warn = FALSE), collapse = "\n")
  srcdoc <- gsub('"', '&quot;', html, fixed = TRUE)
  display_html(sprintf(
    '<iframe srcdoc="%s" width="100%%" height="%dpx" style="border:none;"></iframe>',
    srcdoc, height
  ))
}

## The four assemblies and one alignment

An assembly is the flat `{name, uri}` shorthand — JBrowse picks the adapter from the extension and derives the `.fai`/`.gzi` indexes, so the URL is all you write. One `AllVsAllPAFAdapter` track serves every pair from one PAF file, written as a plain list: the same JSON a JBrowse config file holds.


In [ ]:
base <- "https://jbrowse.org/demos/ecoli_pangenome"
strains <- c("K12", "Sakai", "CFT073", "NCTC86")

assemblies <- lapply(strains, function(s) {
  list(name = s, uri = paste0(base, "/", s, ".fa.gz"))
})

ecoli_ava <- list(
  type = "SyntenyTrack",
  trackId = "ecoli_ava",
  name = "E. coli all-vs-all (minimap2 PAF)",
  assemblyNames = as.list(strains),
  adapter = list(
    type = "AllVsAllPAFAdapter",
    assemblyNames = as.list(strains),
    pafLocation = list(uri = paste0(base, "/all_vs_all.paf.gz"))
  )
)

## Stack them in a synteny view

`synteny_view()` takes the assembly names top-to-bottom. Four rows have three gaps between adjacent pairs, and each gap draws the same all-vs-all track, so `tracks` is that trackId once per gap. `drawCurves = FALSE` draws straight ribbons; `minAlignmentLength` hides short, noisy blocks.


In [ ]:
show_browser(
  JBrowseRApp(
    assemblies = assemblies,
    tracks = list(ecoli_ava),
    views = list(
      synteny_view(
        as.list(strains),
        tracks = list(list("ecoli_ava"), list("ecoli_ava"), list("ecoli_ava")),
        drawCurves = FALSE,
        minAlignmentLength = 10000
      )
    )
  ),
  height = 700
)

## The same PAF as a dotplot

Any one pair opens whole-genome as a dotplot: the long diagonal is the shared backbone, off-diagonal segments are rearrangements.


In [ ]:
show_browser(
  JBrowseRApp(
    assemblies = assemblies[1:2],
    tracks = list(ecoli_ava),
    views = list(dotplot_view(list("K12", "Sakai"), tracks = list("ecoli_ava")))
  ),
  height = 700
)

## Focus each panel on a region

A panel can be a `list(assembly = , loc = )` instead of a bare name, which opens the view already zoomed to that region on each side.


In [ ]:
show_browser(
  JBrowseRApp(
    assemblies = assemblies[1:2],
    tracks = list(ecoli_ava),
    views = list(
      synteny_view(
        list(
          list(assembly = "K12", loc = "chr:1..500,000"),
          list(assembly = "Sakai", loc = "chr:1..500,000")
        ),
        tracks = list(list("ecoli_ava"))
      )
    )
  ),
  height = 700
)

## Next steps

- Two assemblies and a plain `.paf`: `synteny_track(uri, target, query)` builds that track config for you.
- Any other view type a plugin registers: `view(type, ...)` builds the `{type, init}` spec, and `plugins = ` loads the plugin at runtime.
- Building the PAF from your own genomes: the [all-vs-all synteny tutorial](https://jbrowse.org/jb2/docs/tutorials/allvsall_synteny/).

Docs: <https://gmod.github.io/JBrowseR/>
